<a href="https://colab.research.google.com/github/OdysseusPolymetis/atelier_humanistica2026/blob/main/4_topic_modeling_bertopic_grec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Topic modeling de textes grecs anciens avec BERTopic

## C'est quoi, le topic modeling ?

Imaginez : on mélange **500 petits passages** tirés de Platon, Thucydide et Homère, sans étiquette.
On en donne le tas à une machine et on lui demande : *« range ça en groupes thématiques, débrouille-toi »*.
Si elle s'en sort bien, on devrait voir émerger des paquets reconnaissables :
scènes de bataille de l'*Iliade*, dialogues socratiques, harangues politiques de Thucydide…

C'est exactement ce que fait le topic modeling.

Ce qu'il **n'est pas** :
- ce n'est **pas une classification** : personne ne donne d'étiquettes à l'avance, l'ordinateur invente les groupes ;
- ce n'est **pas du RAG** (pas de question posée, pas de réponse générée) ;
- ce n'est **pas un résumé** : on obtient des paquets thématiques, pas un texte synthétique.

## Pourquoi BERTopic plutôt que LDA ?

La méthode classique, **LDA** (Latent Dirichlet Allocation), travaille sur des « sacs de mots » :
elle compte les co-occurrences sans rien comprendre au sens. Sur du grec ancien c'est désastreux :
λέγω, λέγεις, λέγει, ἔλεγον sont vus comme **quatre mots différents**, alors que c'est le même verbe « dire ».

**BERTopic** marche autrement, en quatre étapes :

1. **Embeddings** : chaque passage est transformé en un vecteur de ~768 nombres qui encode son *sens* — pas sa forme.
   Deux passages qui parlent de la guerre auront des vecteurs proches même s'ils utilisent des mots différents.
2. **UMAP** : on réduit ces vecteurs de 768 dimensions à 5 (ou 2 pour visualiser).
3. **HDBSCAN** : on regroupe les points proches les uns des autres en clusters.
4. **c-TF-IDF** : pour chaque cluster, on extrait les mots les plus distinctifs — *ça* devient l'étiquette lisible du topic.

On verra chaque étape en détail.

## 1. Installation

À exécuter une seule fois. `bertopic` tire avec lui sentence-transformers, UMAP et HDBSCAN.
`plotly` sert pour les visualisations interactives.

In [ ]:
!pip install -q \
    bertopic \
    sentence-transformers \
    umap-learn hdbscan \
    plotly matplotlib seaborn \
    pandas numpy scikit-learn tqdm requests \
    "transformers>=4.45" accelerate bitsandbytes

## 2. Imports et configuration

Toutes les constantes utiles sont rassemblées ici. Si vous voulez changer un comportement,
**c'est dans cette cellule** qu'il faut le faire (rarement ailleurs).

In [ ]:
import os
import re
import random
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", DEVICE)

# ------------------------------------------------------------------
# Corpus : on récupère les textes depuis GitHub (rapide, ~3 s).
# C'est la même architecture que dans le notebook RAG : 1 requête
# par édition, pas d'API à interroger 60 fois.
# ------------------------------------------------------------------
GITHUB_CACHE_PATH         = "github_greek_corpus.csv"
GITHUB_GROUP_TARGET_WORDS = 200   # taille cible (en mots) d'un passage

EDITIONS = [
    {"edition_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2", "author": "Homère",     "title": "Iliade"},
    {"edition_urn": "urn:cts:greekLit:tlg0012.tlg002.perseus-grc2", "author": "Homère",     "title": "Odyssée"},
    {"edition_urn": "urn:cts:greekLit:tlg0003.tlg001.perseus-grc2", "author": "Thucydide",  "title": "Guerre du Péloponnèse"},
    {"edition_urn": "urn:cts:greekLit:tlg0059.tlg030.perseus-grc2", "author": "Platon",     "title": "République"},
]

# ------------------------------------------------------------------
# Modèle d'embeddings
# ------------------------------------------------------------------
# SPhilBerta : modèle BERT spécialisé grec ancien & latin (recommandé ici).
# Si vous avez des soucis de chargement, basculez sur LaBSE :
#   EMBEDDING_MODEL_NAME = "sentence-transformers/LaBSE"
EMBEDDING_MODEL_NAME = "bowphs/SPhilBerta"

# ------------------------------------------------------------------
# BERTopic — hyperparamètres principaux
# ------------------------------------------------------------------
MIN_TOPIC_SIZE = 10    # un topic doit contenir au moins N passages
NR_TOPICS      = None  # None = laisser BERTopic décider ; sinon un entier
TOP_N_WORDS    = 10    # nombre de mots-clés affichés par topic

## 3. Récupérer le corpus grec depuis GitHub

Perseus publie ses textes sous forme de fichiers TEI XML uniques par édition sur GitHub.
On télécharge **un fichier entier par auteur** au lieu de 60+ requêtes vers l'API Scaife : c'est ~50× plus rapide.

Le code ci-dessous est identique à celui du notebook RAG. Si vous l'avez déjà exécuté,
le cache `github_greek_corpus.csv` sera réutilisé et il n'y aura **aucun téléchargement** :
vous gagnez 3 secondes mais surtout vous voyez que les données sont les mêmes que dans le RAG.

In [ ]:
PERSEUS_REPOS = [
    ("PerseusDL/canonical-greekLit",     "master"),
    ("OpenGreekAndLatin/First1KGreek",   "master"),
]

def urn_to_relative_path(edition_urn):
    last = edition_urn.split(":")[-1]
    textgroup, work, _ = last.split(".", 2)
    return f"data/{textgroup}/{work}/{last}.xml"


def fetch_tei_xml(edition_urn, timeout=30):
    rel = urn_to_relative_path(edition_urn)
    last_url = None
    for repo, branch in PERSEUS_REPOS:
        url = f"https://raw.githubusercontent.com/{repo}/{branch}/{rel}"
        last_url = url
        try:
            r = requests.get(url, timeout=timeout)
            if r.status_code == 200 and r.text.strip().startswith("<"):
                return r.text, url
        except Exception:
            continue
    raise FileNotFoundError(f"XML introuvable pour {edition_urn} (dernier essai : {last_url})")


def clean_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.replace("\u00a0", " ")
    return re.sub(r"\s+", " ", text).strip()


def parse_tei_leaves(xml_text):
    """Extrait toutes les feuilles `<div type='textpart'>` avec leur référence
    hiérarchique (ex. '1.1.2' = livre.chapitre.section)."""
    for tag in ("note", "bibl", "teiHeader"):
        xml_text = re.sub(rf"<{tag}\b[^>]*>.*?</{tag}>", "", xml_text, flags=re.DOTALL)
    xml_text = re.sub(r'\sxmlns="[^"]+"', "", xml_text, count=1)

    root = ET.fromstring(xml_text)
    parent_of = {child: parent for parent in root.iter() for child in parent}
    all_tps = [e for e in root.iter() if e.tag == "div" and e.get("type") == "textpart"]
    leaf_tps = [tp for tp in all_tps
                if not any(e.tag == "div" and e.get("type") == "textpart"
                           for e in tp.iter() if e is not tp)]
    leaves = []
    for tp in leaf_tps:
        ref_parts, node = [], tp
        while node is not None:
            if node.tag == "div" and node.get("type") == "textpart":
                ref_parts.insert(0, node.get("n", "?"))
            node = parent_of.get(node)
        text = " ".join("".join(tp.itertext()).split())
        if text and ref_parts:
            leaves.append((".".join(ref_parts), text))
    return leaves


def group_leaves_by_words(leaves, target_words=200, max_leaf_words=None):
    """Regroupe les feuilles en chunks d'environ `target_words` mots.
    Pré-découpe les feuilles trop longues (un livre d'Iliade entier, par ex.)."""
    if max_leaf_words is None:
        max_leaf_words = 2 * target_words
    split = []
    for ref, text in leaves:
        words = text.split()
        if len(words) <= max_leaf_words:
            split.append((ref, text))
        else:
            for i in range(0, len(words), target_words):
                sub = " ".join(words[i:i + target_words])
                split.append((f"{ref}#{i // target_words + 1}", sub))
    chunks, current, n_words = [], [], 0
    for ref, text in split:
        current.append((ref, text))
        n_words += len(text.split())
        if n_words >= target_words:
            chunks.append(current)
            current, n_words = [], 0
    if current:
        chunks.append(current)
    return chunks


def build_corpus_from_github_xml(editions, cache_path, target_words=200):
    cache_path = Path(cache_path)
    if cache_path.exists():
        existing = pd.read_csv(cache_path)
        done_urns = set(existing["edition_urn"].dropna().unique())
        print(f"Cache existant : {len(existing)} passages, {len(done_urns)} éditions.")
    else:
        existing, done_urns = pd.DataFrame(), set()

    for edition in editions:
        urn = edition["edition_urn"]
        if urn in done_urns:
            print(f"[skip] {edition.get('author','')} — {edition.get('title','')} (en cache)")
            continue
        try:
            xml_text, url = fetch_tei_xml(urn)
            leaves = parse_tei_leaves(xml_text)
        except Exception as e:
            print(f"  FAIL {urn} : {e}")
            continue
        chunks = group_leaves_by_words(leaves, target_words=target_words)
        print(f"  OK {edition.get('title','')}: {len(leaves)} unités -> {len(chunks)} chunks")

        rows = []
        for block in chunks:
            ref_first, ref_last = block[0][0], block[-1][0]
            reference = ref_first if ref_first == ref_last else f"{ref_first}-{ref_last}"
            rows.append({
                "chunk_id":     f"{urn}:{reference}",
                "edition_urn":  urn,
                "reference":    reference,
                "author":       edition.get("author", ""),
                "title":        edition.get("title", ""),
                "text_greek":   " ".join(b[1] for b in block),
            })
        existing = pd.concat([existing, pd.DataFrame(rows)], ignore_index=True)
        existing.to_csv(cache_path, index=False)

    if existing.empty:
        raise RuntimeError("Aucun texte récupéré.")
    return existing


corpus_df = build_corpus_from_github_xml(
    EDITIONS, cache_path=GITHUB_CACHE_PATH, target_words=GITHUB_GROUP_TARGET_WORDS,
)
corpus_df["text_greek"] = corpus_df["text_greek"].apply(clean_text)
corpus_df = corpus_df[corpus_df["text_greek"].astype(bool)].reset_index(drop=True)
print(f"\nCorpus prêt : {len(corpus_df)} passages")
corpus_df.head()

## 4. Explorer le corpus avant de l'analyser

Règle d'or de l'analyse de données : **avant de lancer un algorithme, regarder ce qu'on a entre les mains**.
On vérifie :

- combien de passages par auteur (équilibre du corpus) ;
- la distribution des longueurs (un passage de 10 mots et un de 500 mots ne portent pas la même information).

Si un auteur est très sur-représenté, BERTopic risque de produire surtout des topics « cet auteur ».
Ce n'est pas forcément un problème — mais il faut le savoir avant d'interpréter.

In [ ]:
corpus_df["n_words"] = corpus_df["text_greek"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

counts = corpus_df.groupby(["author", "title"]).size().reset_index(name="n_chunks")
sns.barplot(data=counts, x="title", y="n_chunks", hue="author", ax=axes[0])
axes[0].set_title("Nombre de passages par œuvre")
axes[0].set_ylabel("passages")
axes[0].tick_params(axis="x", rotation=20)

sns.histplot(data=corpus_df, x="n_words", hue="author", bins=30, ax=axes[1])
axes[1].set_title("Distribution des longueurs (mots)")
axes[1].set_xlabel("mots par passage")

plt.tight_layout()
plt.show()

print("\nRépartition :")
print(counts.to_string(index=False))

## 5. Préparer le texte : stopwords et tokenisation

BERTopic utilise les embeddings (l'étape « sens ») **et** un compteur de mots (l'étape c-TF-IDF qui extrait
les mots distinctifs de chaque topic). C'est ce deuxième composant qu'il faut configurer :
sans filtrage, les topics seraient décrits par des mots comme `καί`, `δέ`, `μέν` qui sont
ultra-fréquents partout et ne disent rien.

On définit donc :

1. une **liste de stopwords grecs anciens** (articles, particules, conjonctions, pronoms, prépositions courantes, formes du verbe « être ») ;
2. un **pattern de token** qui sait reconnaître les caractères grecs (BERTopic utilise par défaut `\w+` qui marche, mais on rend ça explicite pour éviter les surprises) ;
3. un `CountVectorizer` configuré avec ces deux choses, qu'on passera plus tard à BERTopic.

La liste ci-dessous n'est **pas** exhaustive — elle suffit pour ce corpus.
Si vous voyez des mots peu informatifs apparaître dans vos topics finaux, ajoutez-les à `GREEK_STOPWORDS` et relancez la cellule BERTopic.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Stopwords grecs anciens — articles, particules, pronoms, prépositions,
# verbe "être", démonstratifs… Liste à enrichir selon le corpus.
GREEK_STOPWORDS = [
    # articles
    "ὁ", "ἡ", "τό", "οἱ", "αἱ", "τά",
    "τοῦ", "τῆς", "τοῖς", "ταῖς", "τῶν", "τόν", "τήν", "τούς", "τάς",
    "τῷ", "τῇ", "τοὺς", "τὰς", "τὰ", "τὸ", "τὸν", "τὴν",
    # particules et conjonctions
    "καί", "δέ", "μέν", "τε", "γάρ", "οὖν", "ἀλλά", "ἀλλ", "εἰ", "ἢ", "ἤ",
    "δή", "γε", "μή", "μηδέ", "οὐ", "οὐκ", "οὐχ", "οὐδέ", "ὡς", "ὥσπερ",
    "ἔτι", "ἤδη", "νῦν", "νυν", "ποτέ", "τότε", "ἐπεί", "ἐπειδή", "ἵνα", "ὅτι",
    "καὶ", "δὲ", "μὲν", "γὰρ", "τὲ",
    # prépositions
    "ἐν", "εἰς", "ἐκ", "ἐξ", "ἐπί", "κατά", "διά", "μετά", "παρά", "πρός",
    "περί", "ὑπό", "ὑπέρ", "ἀπό", "ἀνά", "σύν", "ἄνευ", "χάριν",
    "ἐπὶ", "διὰ", "μετὰ", "παρὰ", "πρὸς", "περὶ", "ὑπὸ", "ἀπὸ",
    # pronoms démonstratifs / personnels
    "αὐτός", "αὐτή", "αὐτό", "αὐτοῦ", "αὐτῆς", "αὐτῷ", "αὐτῇ", "αὐτόν", "αὐτήν",
    "αὐτοί", "αὐταί", "αὐτά", "αὐτῶν", "αὐτοῖς", "αὐταῖς", "αὐτοὺς", "αὐτάς",
    "οὗτος", "αὕτη", "τοῦτο", "τούτου", "ταύτης", "τούτῳ", "ταύτῃ",
    "τοῦτον", "ταύτην", "οὗτοι", "αὗται", "ταῦτα", "τούτων", "τούτοις",
    "ἐγώ", "σύ", "ἡμεῖς", "ὑμεῖς", "ἐμοῦ", "σοῦ", "ἡμῶν", "ὑμῶν",
    "ἐμοί", "σοί", "ἐμέ", "σέ", "με", "σε", "μοι", "μου", "σου",
    # verbe être (formes les plus courantes)
    "εἰμί", "εἶ", "ἐστί", "ἐστιν", "ἐσμέν", "ἐστέ", "εἰσί", "εἰσίν",
    "ἦν", "ἦσαν", "ἔσται", "εἶναι", "ὤν", "οὖσα", "ὂν", "ὄν",
    "ἐστὶ", "ἐστὶν",
    # divers très fréquents
    "τις", "τι", "τινός", "τινί", "τινά", "τινες", "τινων", "τινῶν",
    "πᾶς", "πᾶσα", "πᾶν", "πάντες", "πάντα", "πάντων", "πάντας",
    "ἄν", "κε", "κεν", "ῥα", "ἄρα", "ἄρ", "ἂν", "ἂρα",
    "ἔφη", "εἶπεν", "εἶπε", "φησί", "φησίν", "φασί",
]

# Token pattern : 2+ lettres (alphabétiques, Unicode-aware) — capture le grec.
# On exclut les tokens d'1 seul caractère qui sont presque toujours du bruit.
GREEK_TOKEN_PATTERN = r"\b[^\W\d_]{2,}\b"

vectorizer = CountVectorizer(
    stop_words=GREEK_STOPWORDS,
    token_pattern=GREEK_TOKEN_PATTERN,
    min_df=2,            # un mot doit apparaître dans >=2 passages pour être gardé
    max_df=0.7,          # un mot dans >70% des passages est trop générique
    ngram_range=(1, 1),  # mots simples ; passez à (1,2) pour aussi capter les bigrammes
)

print(f"{len(GREEK_STOPWORDS)} stopwords définis.")
print("Configuration du vectorizer prête.")

## 6. Charger le modèle d'embeddings

On utilise **SPhilBerta**, un modèle BERT pré-entraîné sur du grec ancien et du latin par l'équipe de l'université de Würzburg.
Comparé à un modèle multilingue généraliste comme LaBSE, il « comprend » beaucoup mieux les nuances du grec classique.

Sentence-transformers transforme automatiquement chaque passage en un vecteur de 768 nombres.
Deux passages au sens proche → vecteurs proches dans l'espace.

In [ ]:
from sentence_transformers import SentenceTransformer

print(f"Chargement de {EMBEDDING_MODEL_NAME} (~500 Mo, peut prendre 1 min la 1re fois)...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)

# On encode tous les passages d'un coup. Sur GPU c'est ~10 s pour ~500 passages.
embeddings = embedding_model.encode(
    corpus_df["text_greek"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
)
print(f"\nMatrice d'embeddings : {embeddings.shape}  (n_passages, n_dimensions)")

## 7. *Avant* de lancer BERTopic : projeter en 2D pour voir

Étape **importante pour comprendre**. BERTopic va clusterer dans un espace réduit (UMAP).
Faisons exactement la même réduction nous-mêmes, mais en 2D, et colorions par auteur.
On voit alors **ce que l'algo va voir** au moment de regrouper.

Ce qu'on cherche à observer :

- les points d'un même auteur forment-ils des paquets cohérents ? (s'il y a structure, BERTopic trouvera des topics)
- y a-t-il des chevauchements entre auteurs ? (= passages thématiquement proches au-delà du style)
- y a-t-il des « îlots » isolés ? (= sujets très particuliers qui formeront probablement un topic à part)

In [ ]:
from umap import UMAP

umap_2d = UMAP(
    n_components=2, n_neighbors=15, min_dist=0.0,
    metric="cosine", random_state=RANDOM_STATE,
)
coords_2d = umap_2d.fit_transform(embeddings)

plot_df = corpus_df.copy()
plot_df["x"] = coords_2d[:, 0]
plot_df["y"] = coords_2d[:, 1]

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=plot_df, x="x", y="y", hue="author", style="title",
    s=40, alpha=0.7,
)
plt.title("Projection UMAP des passages (768 → 2 dimensions)\nchaque point = 1 passage, couleur = auteur")
plt.xlabel(""); plt.ylabel("")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 8. Lancer BERTopic

On peut maintenant exécuter la pipeline complète. BERTopic enchaîne sous le capot :

1. UMAP en **5 dimensions** (pas 2 — 5 préserve mieux la structure pour le clustering, 2 c'est juste pour visualiser) ;
2. HDBSCAN avec `min_cluster_size = MIN_TOPIC_SIZE` ;
3. Vectorisation des passages clusterés (notre `CountVectorizer` configuré plus haut) ;
4. Calcul du c-TF-IDF pour extraire les mots-clés caractéristiques de chaque topic.

On passe nos **embeddings déjà calculés** à BERTopic — il ne refait pas le travail.
L'opération prend quelques secondes.

💡 Le topic `-1` est spécial : c'est le **bac à bruit** d'HDBSCAN. Les passages qui n'appartiennent
clairement à aucun cluster y atterrissent. C'est sain, pas un bug.

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

# On configure explicitement UMAP et HDBSCAN pour avoir un random_state
# (sinon les résultats changent à chaque exécution).
umap_model = UMAP(
    n_components=5, n_neighbors=15, min_dist=0.0,
    metric="cosine", random_state=RANDOM_STATE,
)
hdbscan_model = HDBSCAN(
    min_cluster_size=MIN_TOPIC_SIZE,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    top_n_words=TOP_N_WORDS,
    nr_topics=NR_TOPICS,
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(
    documents=corpus_df["text_greek"].tolist(),
    embeddings=embeddings,
)

corpus_df["topic"] = topics
print(f"\n{topic_model.get_topic_info().shape[0]} topics trouvés (dont topic -1 = bruit)")

## 9. Vue d'ensemble : quels topics, quelle taille ?

`get_topic_info()` donne pour chaque topic : son ID, sa taille, son nom auto-généré (les 3-4 premiers mots-clés).
C'est la première chose à regarder pour avoir une intuition de ce que l'algorithme a trouvé.

In [ ]:
topic_info = topic_model.get_topic_info()
topic_info

## 10. Inspection détaillée : mots-clés par topic

Pour chaque topic, BERTopic calcule un score c-TF-IDF par mot : plus le score est élevé,
plus le mot est **distinctif** de ce topic par rapport aux autres. Ce n'est donc pas une fréquence brute
(sinon `καί` gagnerait toujours, malgré nos stopwords) mais une mesure de spécificité.

Affichons les 10 mots-clés les plus distinctifs pour chacun des topics (en excluant le `-1` bruit).

In [ ]:
def show_topic_keywords(topic_model, top_n=10, exclude_noise=True):
    rows = []
    for tid in sorted(topic_model.get_topics().keys()):
        if exclude_noise and tid == -1:
            continue
        words_scores = topic_model.get_topic(tid)[:top_n]
        words = [w for w, _ in words_scores]
        rows.append({
            "topic": tid,
            "size": int(topic_model.get_topic_info().query('Topic==@tid')["Count"].iloc[0]),
            "mots_clés": " · ".join(words),
        })
    return pd.DataFrame(rows)

show_topic_keywords(topic_model)

## 11. Lire les passages représentatifs

Les mots-clés c'est bien, **mais le test ultime c'est de lire les passages qui composent un topic**.
Si on n'arrive pas à donner un nom de thème en lisant 2-3 passages, c'est que le topic n'est pas net
(soit on a trop de stopwords manquants, soit `MIN_TOPIC_SIZE` est mal réglé).

La fonction ci-dessous prend un topic ID et affiche N passages tirés au hasard dans ce topic
avec leur référence (Homère *Iliade* livre 1.245 etc.).

In [ ]:
def show_topic_examples(topic_id, n=3, max_chars=400):
    subset = corpus_df[corpus_df["topic"] == topic_id]
    if subset.empty:
        print(f"Topic {topic_id} introuvable.")
        return
    sample = subset.sample(min(n, len(subset)), random_state=RANDOM_STATE)
    words = [w for w, _ in topic_model.get_topic(topic_id)[:8]]

    print("=" * 100)
    print(f"Topic {topic_id}  |  {len(subset)} passages")
    print("Mots-clés : " + " · ".join(words))
    print("=" * 100)
    for _, row in sample.iterrows():
        print(f"\n— {row['author']} {row['title']} {row['reference']}")
        text = row["text_greek"]
        print(text[:max_chars] + ("..." if len(text) > max_chars else ""))

# On regarde les 3 plus gros topics non-bruit
for tid in topic_info[topic_info["Topic"] != -1].head(3)["Topic"]:
    show_topic_examples(int(tid), n=3)
    print()

## 12. Visualisations interactives

BERTopic propose plusieurs visualisations Plotly intégrées. Toutes sont **interactives**
(survol, zoom, sélection). Les quatre les plus utiles :

- `visualize_topics()` : carte 2D des topics, distance = (dis)similarité. Survoler = voir mots-clés.
- `visualize_barchart()` : top mots de chaque topic en barres parallèles.
- `visualize_hierarchy()` : arbre de fusion progressive des topics.
- `visualize_heatmap()` : matrice de similarité topic-topic.

Si l'environnement Colab affiche un message du style « Output exceeds the size limit »,
cliquez sur **trust notebook** ou exécutez la cellule isolément.

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_barchart(top_n_topics=12, n_words=8, height=300)

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
topic_model.visualize_heatmap()

## 13. Carte des passages colorés par topic

C'est probablement la visualisation la **plus parlante** pour comprendre ce qui s'est passé.
Chaque point est un passage, sa position est son embedding réduit en 2D, sa couleur est son topic.
On peut survoler pour voir le texte du passage.

À comparer mentalement avec la version « colorée par auteur » qu'on a faite à l'étape 7 :
les topics ne coïncident **pas** parfaitement avec les auteurs — c'est ça l'intérêt du topic modeling,
il découpe selon le contenu, pas selon l'origine.

In [ ]:
topic_model.visualize_documents(
    docs=corpus_df["text_greek"].tolist(),
    embeddings=embeddings,
    hide_annotations=False,
    hide_document_hover=False,
)

## 14. (Bonus) Étiquettes en français générées par Qwen

Lire `μῆνις · θυμός · χόλος · ὀργή` c'est instructif, mais une étiquette française synthétique
(« colère et fureur héroïques ») serait plus parlante. On peut demander ça à Qwen, le LLM utilisé
dans le notebook RAG.

Pour chaque topic, on lui donne les 10 mots-clés grecs **et** 2-3 passages représentatifs,
et on lui demande une étiquette française courte + une glose.

Si vous n'avez pas Qwen installé ou pas de GPU, **sautez cette section** : le reste du notebook
fonctionne sans.

In [ ]:
# Cellule à activer si vous voulez les étiquettes auto.
# Décommenter pour exécuter.

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

QWEN_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

qwen_tok = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
qwen = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    ) if DEVICE == "cuda" else None,
)
qwen.eval()
print("Qwen prêt.")

In [ ]:
@torch.inference_mode()
def qwen_label_topic(topic_id, n_examples=2, max_new_tokens=120):
    """Demande à Qwen une étiquette française courte pour un topic."""
    words = [w for w, _ in topic_model.get_topic(topic_id)[:10]]
    subset = corpus_df[corpus_df["topic"] == topic_id]
    examples = subset.sample(min(n_examples, len(subset)), random_state=RANDOM_STATE)
    examples_str = "\n\n".join(
        f"({r['author']} {r['title']} {r['reference']}) {r['text_greek'][:300]}..."
        for _, r in examples.iterrows()
    )

    user_msg = (
        "Tu es helléniste. Pour le topic suivant, propose UNE étiquette française courte "
        "(3-6 mots max) qui résume le thème, suivie d'une glose d'une phrase.\n\n"
        f"Mots-clés grecs : {', '.join(words)}\n\n"
        f"Exemples de passages :\n\n{examples_str}\n\n"
        "Réponds au format strict :\nÉtiquette : ...\nGlose : ..."
    )
    prompt = qwen_tok.apply_chat_template(
        [{"role": "user", "content": user_msg}],
        tokenize=False, add_generation_prompt=True,
    )
    inputs = qwen_tok(prompt, return_tensors="pt").to(qwen.device)
    out = qwen.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=False, pad_token_id=qwen_tok.eos_token_id,
    )
    gen = out[0, inputs["input_ids"].shape[1]:]
    return qwen_tok.decode(gen, skip_special_tokens=True).strip()


labels = []
for tid in tqdm(topic_info[topic_info["Topic"] != -1]["Topic"], desc="Labellisation"):
    label = qwen_label_topic(int(tid))
    labels.append({"topic": int(tid), "qwen_label": label})

labels_df = pd.DataFrame(labels)
for _, row in labels_df.iterrows():
    print(f"\n=== Topic {row['topic']} ===")
    print(row["qwen_label"])